# Import Library & Import Data

In [1]:
# Data Processing
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Datetime Standardize
import datetime

In [2]:
# Import Library & Import Data
import sys
import matplotlib

print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")
print(f"numpy version: {np.__version__}")

Python version: 3.13.16
pandas version: 2.2.3
matplotlib version: 3.10.0
numpy version: 2.1.3


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Tải các bảng dữ liệu
page_view = pd.read_csv('/content/drive/MyDrive/K04 - Scripts các buổi học/Dataset/Buổi 8: Funnel Chart/website_pageviews.csv')
session = pd.read_csv('/content/drive/MyDrive/K04 - Scripts các buổi học/Dataset/Buổi 8: Funnel Chart/website_sessions.csv')

print("Dữ liệu đã sẵn sàng!")

Dữ liệu đã sẵn sàng!


# Data Checking

# Data Preprocessing / Wrangling

In [4]:
page_view.head(2)

,website_pageview_id,created_at,website_session_id,pageview_url
0,1,2012-03-19 08:04:16,1,/home
1,2,2012-03-19 08:16:49,2,/home


In [5]:
session.head(2)

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
0,1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
1,2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com


# Exploratory - EDA (Đưa vào phương pháp phân tích or thống kê)

# Câu 1: Xây dựng Phễu chuyển đổi cơ bản (Basic Funnel)

Hãy xác định có bao nhiêu % khách hàng đi qua các bước sau trong khoảng thời gian từ 2012-08-05 đến 2012-09-05:

- Trang chủ: /home

- Trang sản phẩm: /products

- Trang chi tiết sản phẩm: /the-original-mr-fuzzy

- Giỏ hàng: /cart

- Thông tin vận chuyển: /shipping

- Thông tin thanh toán: /billing

- Hoàn tất đơn hàng: /thank-you

Yêu cầu: Tính số lượng session đạt được ở mỗi bước và tỷ lệ duy trì (click-through rate) giữa các bước kế tiếp.

In [6]:
session_merge = session.merge(page_view, on = 'website_session_id', how = 'left')

session_merge.head(2)

,website_session_id,created_at_x,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer,website_pageview_id,created_at_y,pageview_url
0,1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,1,2012-03-19 08:04:16,/home
1,2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,2,2012-03-19 08:16:49,/home


In [8]:
session_merge_filter = session_merge[session_merge['created_at_y'].between('2012-08-05', '2012-09-05')].reset_index(drop = True)

session_merge_filter.head(5)

,website_session_id,created_at_x,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer,website_pageview_id,created_at_y,pageview_url
0,18243,2012-08-05 00:11:26,16827,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,37861,2012-08-05 00:11:26,/lander-1
1,18244,2012-08-05 00:44:06,16828,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,37862,2012-08-05 00:44:06,/lander-1
2,18244,2012-08-05 00:44:06,16828,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,37863,2012-08-05 00:46:01,/products
3,18244,2012-08-05 00:44:06,16828,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,37864,2012-08-05 00:46:32,/the-original-mr-fuzzy
4,18244,2012-08-05 00:44:06,16828,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,37866,2012-08-05 00:47:26,/cart


In [10]:
session_merge_filter.groupby(by = 'pageview_url', as_index = False).agg(count_customer = ('user_id' , 'nunique') ,

                                                                 count_session = ('website_session_id', 'nunique')

).sort_values(by = 'count_customer', ascending = False)

,pageview_url,count_customer,count_session
3,/lander-1,5269,5269
4,/products,2890,2946
7,/the-original-mr-fuzzy,2124,2150
1,/cart,937,942
2,/home,708,783
5,/shipping,631,634
0,/billing,493,496
6,/thank-you-for-your-order,214,214


In [11]:
session_with_home_list = page_view[page_view['pageview_url'] == '/home']['website_session_id'].drop_duplicates().to_list()

session_merge_filter_home = session_merge_filter[session_merge_filter['website_session_id'].isin(session_with_home_list)]

session_merge_filter_home.head()

,website_session_id,created_at_x,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer,website_pageview_id,created_at_y,pageview_url
13,18248,2012-08-05 01:30:19,16729,1,NaN,NaN,NaN,desktop,https://www.bsearch.com,37874,2012-08-05 01:30:19,/home
14,18248,2012-08-05 01:30:19,16729,1,NaN,NaN,NaN,desktop,https://www.bsearch.com,37875,2012-08-05 01:33:24,/products
26,18253,2012-08-05 04:44:44,16836,0,NaN,NaN,NaN,desktop,https://www.gsearch.com,37887,2012-08-05 04:44:44,/home
27,18253,2012-08-05 04:44:44,16836,0,NaN,NaN,NaN,desktop,https://www.gsearch.com,37888,2012-08-05 04:49:59,/products
85,18284,2012-08-05 11:33:17,8650,1,NaN,NaN,NaN,mobile,NaN,37944,2012-08-05 11:33:17,/home


In [12]:
session_merge_filter_home.groupby(by = 'pageview_url', as_index = False).agg(count_customer = ('user_id' , 'nunique') ,

                                                                 count_session = ('website_session_id', 'nunique')

).sort_values(by = 'count_customer', ascending = False)

,pageview_url,count_customer,count_session
2,/home,708,783
3,/products,453,482
6,/the-original-mr-fuzzy,322,336
1,/cart,155,159
4,/shipping,115,118
0,/billing,83,86
5,/thank-you-for-your-order,33,33


In [13]:
# 1. Lưu kết quả groupby vào một DataFrame
df_funnel = (
    session_merge_filter_home.groupby(by='pageview_url', as_index=False)
    .agg(count_customer=('user_id', 'nunique'))
    .sort_values(by='count_customer', ascending=False) # Lưu ý: Đảm bảo thứ tự dòng là thứ tự các bước trong phễu
)

# 2. Tính CR 1st stage (So với bước đầu tiên)
first_stage_count = df_funnel['count_customer'].iloc[0]
df_funnel['CR 1st stage'] = df_funnel['count_customer'] / first_stage_count

# 3. Tính CR prev stage (So với bước liền trước)
df_funnel['CR prev stage'] = df_funnel['count_customer'] / df_funnel['count_customer'].shift(1)

# Xử lý giá trị NaN cho dòng đầu tiên ở cột CR prev stage (đặt thành 1.0 hoặc 100%)
df_funnel['CR prev stage'] = df_funnel['CR prev stage'].fillna(1.0)

# (Tùy chọn) Chuyển đổi sang định dạng phần trăm (%) nếu muốn dễ nhìn hơn
df_funnel['CR 1st stage (%)'] = (df_funnel['CR 1st stage'] * 100).round(2).astype(str) + '%'
df_funnel['CR prev stage (%)'] = (df_funnel['CR prev stage'] * 100).round(2).astype(str) + '%'

df_funnel

,pageview_url,count_customer,CR 1st stage,CR prev stage,CR 1st stage (%),CR prev stage (%)
2,/home,708,1.000000,1.000000,100.0%,100.0%
3,/products,453,0.639831,0.639831,63.98%,63.98%
6,/the-original-mr-fuzzy,322,0.454802,0.710817,45.48%,71.08%
1,/cart,155,0.218927,0.481366,21.89%,48.14%
4,/shipping,115,0.162429,0.741935,16.24%,74.19%
0,/billing,83,0.117232,0.721739,11.72%,72.17%
5,/thank-you-for-your-order,33,0.046610,0.397590,4.66%,39.76%


# Feature Engineering (Optional)

# Conclusion & Recommendation